# Borzoi / Flashzoi: Prediction + Benchmarking

Этот ноутбук объединяет:

- предсказание экспрессии из `run_borzoi_prediction_pytorch.py`
- benchmark / корреляции / графики из `only_benchmarking_clean.ipynb`

В notebook теперь есть переключатель:

- `model_variant = "borzoi"`
- `model_variant = "flashzoi"`

Логика разбита на шаги:

1. конфигурация и пути
2. загрузка модели и аннотаций
3. функции для инференса по генам
4. запуск предсказания и сохранение `csv`
5. benchmark против `ground truth`
6. визуализация по таргетам


In [ ]:
import os
import sys
from contextlib import nullcontext
from typing import Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from pyfaidx import Fasta
from borzoi_pytorch import Borzoi, Transcriptome

# ----------------------------
# Основная конфигурация
# ----------------------------
split = "valid"  # "test" или "valid"
model_variant = "borzoi"  # "borzoi" или "flashzoi"
replicate = 0
run_prediction = True

# Если True, перед агрегацией по экзонам предсказания будут переведены
# из squashed scale обратно в coverage-like scale по параметрам из targets_human.txt.
undo_track_transform = True

# Что использовать как ground truth для benchmark-а:
# "main"     -> {split}_true_human.csv
# "tcell_bw" -> borzoi_ground_true_from_bw_T-cell.csv
true_source = "main"

# Если хочешь ограничить число строк при отладке, поставь integer.
debug_max_rows = None

# Если хочешь рисовать только часть таргетов, задай список id.
plot_targets = None

HOME = '/home/jovyan/shares/SR003.nfs2/aspeedok/'
# ----------------------------
# Пути к данным
# ----------------------------
BORZOI_DIR = os.path.join(HOME, "GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi")

FASTA_PATH = os.path.join(HOME, "GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/hg38.fa")
TARGETS_FILE = os.path.join(BORZOI_DIR, "targets_human.txt")
ANNOT_GTF = os.path.join(BORZOI_DIR, "gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz")
# MAPPING_PATH оставлен только как reference; intersection mode ниже его не использует.
# SELECTED_TARGETS_PATH = os.path.join(BORZOI_DIR, "selected_targets.csv")

genes_forward_path = os.path.join(BORZOI_DIR, f"human.{split}.forward.csv")
genes_reverse_path = os.path.join(BORZOI_DIR, f"human.{split}.reverse.csv")

true_main_path = os.path.join(BORZOI_DIR, f"{split}_true_human.csv")
# true_tcell_bw_path = "/data/ddpanchenko/main_dom/GENA/Borzoi/predictions/22012026/borzoi_ground_true_from_bw_T-cell.csv"
SCORE_SRC_DIR = os.path.join(HOME, "GENA_LM/downstream_tasks/expression_prediction/datasets/src")

# ----------------------------
# Параметры модели / устройства
# ----------------------------
SEQ_LEN = 524_288
MODEL_STRIDE = 32

# Это рабочая гипотеза для привязки координат PyTorch-выхода к геному.
# Значение взято из upstream notebook borzoi-pytorch, где TF-предсказания режут
# как [..., 5104:-5104], чтобы получить тот же центральный выход, что и в PyTorch.
# Для Flashzoi пока используем ту же привязку, но это место стоит валидировать отдельно.
MODEL_CROP_BINS = 5120

# Sanity check: ожидаем длину выхода PyTorch Borzoi/Flashzoi в 6144 бинов.
EXPECTED_OUTPUT_BINS = 6144

DEVICE = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

model_name_map = {
    "borzoi": f"johahi/borzoi-replicate-{replicate}",
    "flashzoi": f"johahi/flashzoi-replicate-{replicate}",
}
if model_variant not in model_name_map:
    raise ValueError(f"Unknown model_variant: {model_variant}")

model_name = model_name_map[model_variant]
use_flashzoi = model_variant == "flashzoi"
use_autocast = use_flashzoi

if use_flashzoi and DEVICE.type != "cuda":
    raise RuntimeError(
        "Flashzoi в upstream README требует modern Nvidia GPU и autocast. "
        "На CPU в этом notebook он отключён."
    )

if DEVICE.type == "cuda":
    autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    autocast_dtype = None

pred_out_path = os.path.join(
    BORZOI_DIR,
    f"{model_variant}_predictions_{split}_replicate{replicate}_pytorch.csv",
)

print("Model variant:", model_variant)
print("Model name:", model_name)
print("Autocast enabled:", use_autocast)
print("Autocast dtype:", autocast_dtype)
print("Undo track transform:", undo_track_transform)


ModuleNotFoundError: No module named 'borzoi_pytorch'

In [4]:
# Широкая qnorm ground truth: столбцы = id (ENCFF) из обоих маппингов _qnorm.
# Один и тот же id (ENCFF файла TPM): приоритет строки из file_mappings_borzoi_human_qnorm (keep="last").

FILE_MAPPINGS_DIR = os.path.normpath(
    os.path.join(BORZOI_DIR, "..", "..", "datasets", "data", "file_mappings")
)
EXPR_MAP_QNORM = os.path.join(FILE_MAPPINGS_DIR, "Expression_dataset_v1_csv_file_mappings_qnorm.csv")
BORZOI_MAP_QNORM = os.path.join(FILE_MAPPINGS_DIR, "file_mappings_borzoi_human_qnorm.csv")

expr_map = pd.read_csv(EXPR_MAP_QNORM)
borzoi_map = pd.read_csv(BORZOI_MAP_QNORM)
samples_meta = pd.concat([expr_map, borzoi_map], ignore_index=True).drop_duplicates(
    subset=["id"], keep="last"
)


def resolve_tpm_path(csv_rel: str) -> str:
    return os.path.normpath(os.path.join(FILE_MAPPINGS_DIR, csv_rel))


out_true_wide_path = os.path.join(BORZOI_DIR, f"{split}_true_human_all_qnorm_id.csv")

series_list = []
missing_files = []
for _, row in tqdm(samples_meta.iterrows(), total=len(samples_meta), desc="Load qnorm TPM"):
    tpm_path = resolve_tpm_path(row["csv"])
    if not os.path.isfile(tpm_path):
        missing_files.append(tpm_path)
        continue
    wide = pd.read_csv(tpm_path, nrows=1)
    col_name = str(row["id"])
    ser = pd.Series(wide.iloc[0].to_numpy(), index=wide.columns.astype(str), name=col_name)
    series_list.append(ser)

if missing_files:
    raise FileNotFoundError(
        f"Нет {len(missing_files)} TPM (первые 5): {missing_files[:5]}"
    )

wide_true_df = pd.concat(series_list, axis=1, join="outer")
wide_true_df.insert(0, "gene_id", wide_true_df.index.to_numpy())
wide_true_df = wide_true_df.reset_index(drop=True)

# Ограничить гены сплита как в benchmark (число строк как у valid_true_human): поставь True
restrict_to_split_genes = False
if restrict_to_split_genes:
    bw_f = pd.read_csv(genes_forward_path, sep="\t")
    bw_r = pd.read_csv(genes_reverse_path, sep="\t")
    g_ok = set(pd.concat([bw_f, bw_r], ignore_index=True)["gene_id"].astype(str))
    wide_true_df = wide_true_df[wide_true_df["gene_id"].astype(str).isin(g_ok)].reset_index(drop=True)

wide_true_df.to_csv(out_true_wide_path, index=False)
print("Saved", wide_true_df.shape, "->", out_true_wide_path)

Load qnorm TPM:   0%|          | 0/815 [00:00<?, ?it/s]

Load qnorm TPM:   1%|▏         | 11/815 [00:03<04:32,  2.96it/s]


KeyboardInterrupt: 

In [5]:
genome = Fasta(FASTA_PATH)

# --- qnorm id/original_id <-> tracks в targets_human (полный mapping, без MAPPING_PATH) ---
FILE_MAPPINGS_DIR = os.path.normpath(
    os.path.join(BORZOI_DIR, "..", "..", "datasets", "data", "file_mappings")
)
EXPR_MAP_QNORM = os.path.join(FILE_MAPPINGS_DIR, "Expression_dataset_v1_csv_file_mappings_qnorm.csv")
BORZOI_MAP_QNORM = os.path.join(FILE_MAPPINGS_DIR, "file_mappings_borzoi_human_qnorm.csv")
expr_map_q = pd.read_csv(EXPR_MAP_QNORM)
borzoi_map_q = pd.read_csv(BORZOI_MAP_QNORM)
qnorm_target_id_map_df = (
    pd.concat([expr_map_q, borzoi_map_q], ignore_index=True)
    .drop_duplicates(subset=["id"], keep="last")[["id", "original_id"]]
    .copy()
)
qnorm_target_id_map_df["id"] = qnorm_target_id_map_df["id"].astype(str)
qnorm_target_id_map_df["original_id"] = qnorm_target_id_map_df["original_id"].astype(str)

targets_ref = pd.read_csv(TARGETS_FILE, sep="\t", index_col=0).reset_index(drop=True)
targets_ref["identifier_base"] = targets_ref["identifier"].astype(str).str.replace(
    r"[+-]$", "", regex=True
)
targets_ref["file"] = targets_ref["file"].astype(str)

TPM_TO_TARGETS_TRACK = {
    "ENCFF035CWS": "ENCFF387UUZ",
    "ENCFF761SPP": "ENCFF917RKL",
}


def _resolve_targets_identifier_base(row: pd.Series):
    tid = str(row["id"])
    if tid in TPM_TO_TARGETS_TRACK:
        return TPM_TO_TARGETS_TRACK[tid]
    oid = str(row["original_id"])
    for acc in (oid, tid):
        m = targets_ref.loc[targets_ref["identifier_base"] == acc]
        if len(m):
            return m.iloc[0]["identifier_base"]
    for acc in (oid, tid):
        m = targets_ref.loc[targets_ref["file"].str.contains(acc, regex=False, na=False)]
        if len(m):
            return m.iloc[0]["identifier_base"]
    return pd.NA


qnorm_target_id_map_df["targets_identifier_base"] = qnorm_target_id_map_df.apply(
    _resolve_targets_identifier_base, axis=1
)
_miss_map = int(qnorm_target_id_map_df["targets_identifier_base"].isna().sum())
if _miss_map:
    print("[WARN] qnorm map: нет targets_identifier_base для строк:", _miss_map)

qnorm_target_id_map_path = os.path.join(BORZOI_DIR, "qnorm_id_to_targets_map.csv")
qnorm_target_id_map_df.to_csv(qnorm_target_id_map_path, index=False)

print("Saved full qnorm -> targets_human map:", qnorm_target_id_map_path)
print("Rows in qnorm_target_id_map_df:", len(qnorm_target_id_map_df))
print("Unique qnorm id:", qnorm_target_id_map_df["id"].nunique())
print("Unique original_id:", qnorm_target_id_map_df["original_id"].nunique())
qnorm_target_id_map_df.head()


Saved full qnorm -> targets_human map: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/qnorm_id_to_targets_map.csv
Rows in qnorm_target_id_map_df: 815
Unique qnorm id: 815
Unique original_id: 815


,id,original_id,targets_identifier_base
0,ENCFF035CWS,ENCSR094GVZ,ENCFF387UUZ
5,ENCFF329ENM,ENCFF672VYQ,ENCFF672VYQ
11,ENCFF761SPP,ENCSR561FEE,ENCFF917RKL
14,ENCFF731CJY,ENCFF168OLY,ENCFF168OLY
15,ENCFF242NRP,ENCFF153YEN,ENCFF153YEN


In [ ]:
# --- пересечение qnorm mapping с targets_human: индексы выходов модели и track transforms ---
targets_df = pd.read_csv(TARGETS_FILE, sep="	", index_col=0).reset_index(drop=True)
targets_df["target_row"] = np.arange(len(targets_df), dtype=int)
targets_df["identifier_base"] = targets_df["identifier"].astype(str).str.replace(
    r"[+-]$", "", regex=True
)

qnorm_targets_intersection_df = (
    qnorm_target_id_map_df
    .dropna(subset=["targets_identifier_base"])
    [["id", "original_id", "targets_identifier_base"]]
    .drop_duplicates()
    .copy()
)

targets_df_sub = targets_df.merge(
    qnorm_targets_intersection_df,
    left_on="identifier_base",
    right_on="targets_identifier_base",
    how="inner",
).copy()

target_index_sub = targets_df_sub["target_row"].to_numpy(dtype=int)
target_identifiers = targets_df_sub["identifier"].astype(str).tolist()

target_scale = targets_df_sub["scale"].to_numpy(dtype=np.float32)
target_clip = targets_df_sub["clip"].to_numpy(dtype=np.float32)
target_clip_soft = targets_df_sub["clip_soft"].to_numpy(dtype=np.float32)
target_sum_stat = targets_df_sub["sum_stat"].astype(str).to_numpy()
target_transform = np.where(target_sum_stat == "sum_sqrt", 3.0 / 4.0, 1.0).astype(np.float32)

target_params_df = targets_df_sub[
    [
        "identifier",
        "identifier_base",
        "id",
        "original_id",
        "clip",
        "clip_soft",
        "scale",
        "sum_stat",
        "description",
    ]
].copy()

qnorm_targets_intersection_path = os.path.join(BORZOI_DIR, "qnorm_targets_human_intersection.csv")
target_params_df.to_csv(qnorm_targets_intersection_path, index=False)

print("Intersected target tracks (+/- rows):", len(target_index_sub))
print("Unique original_id in intersection:", targets_df_sub["original_id"].nunique())
print("Unique qnorm id in intersection:", targets_df_sub["id"].nunique())
print("Saved intersection table:", qnorm_targets_intersection_path)
print("Unique target transform configs:")
print(
    target_params_df[["clip", "clip_soft", "scale", "sum_stat"]]
    .drop_duplicates()
    .sort_values(["clip", "clip_soft", "scale", "sum_stat"])
    .to_string(index=False)
)


Intersected target tracks (+/- rows): 1373
Unique original_id in intersection: 815
Unique qnorm id in intersection: 815
Saved intersection table: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/qnorm_targets_human_intersection.csv
Unique target transform configs:
 clip  clip_soft    scale sum_stat
  768        384 0.300000 sum_sqrt
  768        384 0.312422 sum_sqrt
  768        384 0.348014 sum_sqrt
  768        384 0.427742 sum_sqrt
  768        384 0.639003 sum_sqrt
  768        384 0.730714 sum_sqrt
  768        384 0.873491 sum_sqrt
  768        384 2.003142 sum_sqrt


In [ ]:
target_params_df[['identifier'] == 'ENCFF035CWS']

In [10]:
tmp = targets_df_sub.copy()
tmp["strand"] = tmp["identifier"].astype(str).str.extract(r"([+-])$")[0]
tmp["strand"] = tmp["strand"].fillna("no_suffix")

strand_counts = (
    tmp.groupby(["id", "original_id", "identifier_base", "strand"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["+", "-", "no_suffix"]:
    if col not in strand_counts.columns:
        strand_counts[col] = 0

strand_counts["n_rows"] = strand_counts["+"] + strand_counts["-"] + strand_counts["no_suffix"]

print(strand_counts["n_rows"].value_counts().sort_index())
print()
print("Rows without +/- suffix:", (strand_counts["no_suffix"] > 0).sum())

display(
    strand_counts[strand_counts["no_suffix"] > 0]
    .sort_values(["id", "original_id"])
)


n_rows
1    257
2    558
Name: count, dtype: int64

Rows without +/- suffix: 257


strand,id,original_id,identifier_base,+,-,no_suffix,n_rows
0,ENCFF003KHL,ENCFF860TYK,ENCFF860TYK,0,0,1,1
11,ENCFF014BZI,ENCFF367NBV,ENCFF367NBV,0,0,1,1
16,ENCFF016NWL,ENCFF946QQD,ENCFF946QQD,0,0,1,1
18,ENCFF020OPI,ENCFF681MQA,ENCFF681MQA,0,0,1,1
19,ENCFF020WAT,ENCFF354WKE,ENCFF354WKE,0,0,1,1
...,...,...,...,...,...,...,...
794,ENCFF982FSG,ENCFF609GRY,ENCFF609GRY,0,0,1,1
795,ENCFF986KFO,ENCFF246RCD,ENCFF246RCD,0,0,1,1
796,ENCFF986ZVF,ENCFF065NBH,ENCFF065NBH,0,0,1,1
805,ENCFF993TGD,ENCFF848ZVQ,ENCFF848ZVQ,0,0,1,1


In [11]:
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

model = Borzoi.from_pretrained(model_name)
model.to(DEVICE)
model.eval()

print(f"Loaded {model_name} on {DEVICE}")

Loaded johahi/borzoi-replicate-0 on cuda:6


In [12]:
transcriptome_by_id = Transcriptome(ANNOT_GTF, use_geneid=True)
transcriptome_by_name = Transcriptome(ANNOT_GTF, use_geneid=False)

In [13]:
transcriptome_by_name.genes

{'DDX11L1': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe31a610>,
 'WASH7P': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe31b490>,
 'MIR6859-1': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe31ba90>,
 'MIR1302-2HG': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe31bf50>,
 'MIR1302-2': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe325690>,
 'FAM138A': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe325810>,
 'OR4G4P': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe326b90>,
 'OR4G11P': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe326390>,
 'OR4F5': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe327ad0>,
 'AL627309.1': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe327fd0>,
 'AL627309.3': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe32d3d0>,
 'CICP27': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe32da50>,
 'AL627309.6': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe32d350>,
 'AL627309.7': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe32e350>,
 'AL627309.2': <borzoi_pytorch.gene_utils.Gene at 0x7efbbe32edd0>,
 'AL627309.5': <borzoi

In [14]:
transcriptome_by_id.genes

{'ENSG00000223972.5': <borzoi_pytorch.gene_utils.Gene at 0x7efe6bdaac90>,
 'ENSG00000227232.5': <borzoi_pytorch.gene_utils.Gene at 0x7efe6c00dfd0>,
 'ENSG00000278267.1': <borzoi_pytorch.gene_utils.Gene at 0x7efe6bdab850>,
 'ENSG00000243485.5': <borzoi_pytorch.gene_utils.Gene at 0x7efe68525810>,
 'ENSG00000284332.1': <borzoi_pytorch.gene_utils.Gene at 0x7efe68526810>,
 'ENSG00000237613.2': <borzoi_pytorch.gene_utils.Gene at 0x7efe68527150>,
 'ENSG00000268020.3': <borzoi_pytorch.gene_utils.Gene at 0x7efe68526110>,
 'ENSG00000240361.2': <borzoi_pytorch.gene_utils.Gene at 0x7efe68527dd0>,
 'ENSG00000186092.6': <borzoi_pytorch.gene_utils.Gene at 0x7efe693c7ad0>,
 'ENSG00000238009.6': <borzoi_pytorch.gene_utils.Gene at 0x7efeb086c610>,
 'ENSG00000239945.1': <borzoi_pytorch.gene_utils.Gene at 0x7efe68fdfb10>,
 'ENSG00000233750.3': <borzoi_pytorch.gene_utils.Gene at 0x7effa8203150>,
 'ENSG00000268903.1': <borzoi_pytorch.gene_utils.Gene at 0x7efe6c6c1a90>,
 'ENSG00000269981.1': <borzoi_pytorch.

In [9]:
if 'AC016700' in transcriptome_by_name.genes:
    print('yes')
else:
    print('no')
 

no


In [15]:
def one_hot_encode_dna(seq: str) -> np.ndarray:
    mapping = {
        "A": [1, 0, 0, 0],
        "C": [0, 1, 0, 0],
        "G": [0, 0, 1, 0],
        "T": [0, 0, 0, 1],
        "N": [0, 0, 0, 0],
    }
    arr = np.zeros((len(seq), 4), dtype="float32")
    for i, base in enumerate(seq.upper()):
        arr[i] = mapping.get(base, mapping["N"])
    return arr


def get_window_sequence(row, seq_len=SEQ_LEN):
    chrom = row["chrom"]
    tss = int(row["TSS"])
    strand = row["gene_strand"]

    half = seq_len // 2
    start = tss - half
    if start < 1:
        start = 1
    end = start + seq_len

    seq = genome[chrom][start:end].seq
    if len(seq) < seq_len:
        seq = seq + "N" * (seq_len - len(seq))
    elif len(seq) > seq_len:
        seq = seq[:seq_len]

    if strand == "-":
        comp = str.maketrans("ACGTacgt", "TGCAtgca")
        seq = seq.translate(comp)[::-1]

    return seq, start


p = 0


def find_gene_obj(row):
    global p

    gid = str(row["gene_id_unversioned"])
    if gid in transcriptome_by_id.genes:
        return transcriptome_by_id.genes[gid]

    gname = str(row["gene_name"])
    if gname in transcriptome_by_name.genes:
        return transcriptome_by_name.genes[gname]

    p += 1
    raise ValueError(f"Gene not found in transcriptome: {gid} / {gname}")


def get_autocast_context():
    if use_autocast and DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=autocast_dtype)
    return nullcontext()


def undo_track_transform_from_targets(pred_2d: np.ndarray) -> np.ndarray:
    # pred_2d: [n_bins, n_targets_sub] в squashed scale
    x = pred_2d.astype(np.float32).copy()

    # 1. Убираем track-specific scale
    x = x / target_scale[None, :]

    # 2. Убираем soft squash по track-specific clip_soft
    clip_soft = target_clip_soft[None, :]
    mask = x > clip_soft
    x = np.where(mask, (x - clip_soft) ** 2 + clip_soft, x)

    # 3. Обращаем степень:
    #    sum_sqrt -> transform 3/4 -> inverse 4/3
    #    sum      -> transform 1.0 -> inverse 1.0
    x = x ** (1.0 / target_transform[None, :])
    return x


def predict_one_sequence(one_hot_seq: np.ndarray) -> np.ndarray:
    x = torch.from_numpy(one_hot_seq).permute(1, 0).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        with get_autocast_context():
            y = model(x)

    pred = y[0].detach().float().cpu().numpy().transpose(1, 0)

    if pred.shape[0] != EXPECTED_OUTPUT_BINS:
        raise ValueError(
            f"Unexpected number of output bins: {pred.shape[0]} "
            f"(expected {EXPECTED_OUTPUT_BINS})"
        )

    pred = pred[:, target_index_sub]

    if undo_track_transform:
        pred = undo_track_transform_from_targets(pred)

    return pred


def gene_vector_for_row(row) -> Optional[np.ndarray]:
    seq, seq_start = get_window_sequence(row)
    one_hot = one_hot_encode_dna(seq)
    pred = predict_one_sequence(one_hot)

    if row["gene_strand"] == "-":
        pred = pred[::-1, :]

    gene_obj = find_gene_obj(row)

    seq_out_start = seq_start + MODEL_STRIDE * MODEL_CROP_BINS
    seq_out_len = MODEL_STRIDE * pred.shape[0]

    gene_slice = gene_obj.output_slice(
        seq_out_start,
        seq_out_len,
        MODEL_STRIDE,
        False,
    )

    if len(gene_slice) == 0:
        print(f"Gene has empty output slice: {row['gene_id']}")
        return None

    # В этом notebook считаем средний сигнал по экзонным бинам proxy для gene expression.
    exon_pred = pred[gene_slice, :]
    expr = exon_pred.mean(axis=0)

    return expr.astype(np.float32)


def run_split(forward_path, reverse_path, out_csv_path, max_rows=None):
    print(f"\n=== Run split ===\n{forward_path}\n{reverse_path}\n=> {out_csv_path}")

    df_f = pd.read_csv(forward_path, sep="\t")
    df_f = df_f
    df_r = pd.read_csv(reverse_path, sep="\t")
    df_r = df_r

    df = pd.concat([df_f, df_r], ignore_index=True)
    if max_rows is not None:
        df = df.iloc[:max_rows].copy()

    print("Total genes (forward+reverse):", len(df))

    gene_to_vecs = {}

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            vec = gene_vector_for_row(row)
        except Exception as e:
            print(f"  [WARN] пропускаем строку {idx} ({row['gene_id']}): {e}")
            continue

        if vec is None:
            continue

        gid = str(row["gene_id"])
        gene_to_vecs.setdefault(gid, []).append(vec)

    genes = []
    preds = []
    for gid, vec_list in gene_to_vecs.items():
        arr = np.stack(vec_list, axis=0)
        mean_vec = arr.mean(axis=0)
        genes.append(gid)
        preds.append(mean_vec)

    preds = np.stack(preds, axis=0)

    # intermediate columns = identifier (+/-) из targets_human; после melt+pivot — original_id.
    out_df = pd.DataFrame(preds, columns=target_identifiers)
    out_df.insert(0, "gene_id", genes)

    merged = out_df.melt(
        id_vars=["gene_id"],
        value_vars=target_identifiers,
        var_name="identifier",
        value_name="expr",
    )

    merged = merged.merge(
        targets_df_sub[["identifier", "original_id"]],
        on="identifier",
        how="left",
    )

    final = merged.groupby(["gene_id", "original_id"])["expr"].sum().reset_index()
    final_df = final.pivot(index="gene_id", columns="original_id", values="expr").reset_index()
    final_df.to_csv(out_csv_path, index=False)

    print("Done:", final_df.shape)
    print("Missing genes count:", p)
    return final_df


## Шаг 1. Предсказание PyTorch Borzoi / Flashzoi

Если `run_prediction = True`, ячейка ниже прогонит выбранную модель по всем генам
и сохранит результат в `pred_out_path`.

Варианты:

- `model_variant = "borzoi"`: обычный PyTorch-порт Borzoi
- `model_variant = "flashzoi"`: ускоренная версия; в этом notebook для неё автоматически включается `autocast`

Если `run_prediction = False`, ячейка просто загрузит уже сохранённый `csv`.


In [16]:
if run_prediction:
    pred_gene_df = run_split(
        genes_forward_path,
        genes_reverse_path,
        pred_out_path,
        max_rows=debug_max_rows,
    )
else:
    pred_gene_df = pd.read_csv(pred_out_path)

print(pred_out_path)
print("p =", p)
pred_gene_df.head()

pred_all_qnorm_path = os.path.join(
    BORZOI_DIR,
    f"{split}_pred_human_all_qnorm_id.csv",
)
pred_gene_df.to_csv(pred_all_qnorm_path, index=False)
print("Saved aligned pred table ->", pred_all_qnorm_path)



=== Run split ===
/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/human.valid.forward.csv
/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/human.valid.reverse.csv
=> /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/borzoi_predictions_valid_replicate0_pytorch.csv
Total genes (forward+reverse): 3038


  0%|          | 0/3038 [00:00<?, ?it/s]

 46%|████▌     | 1402/3038 [20:28<23:43,  1.15it/s]

Gene has empty output slice: ENSG00000178591.6


 46%|████▌     | 1403/3038 [20:29<23:40,  1.15it/s]

...

 87%|████████▋ | 2642/3038 [38:37<05:47,  1.14it/s]

Gene has empty output slice: ENSG00000249054.2


100%|██████████| 3038/3038 [44:28<00:00,  1.14it/s]


Done: (3032, 816)
Missing genes count: 0
/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/borzoi_predictions_valid_replicate0_pytorch.csv
p = 0


AttributeError: 'str' object has no attribute 'to_csv'

In [17]:
pred_gene_df

original_id,gene_id,ENCFF004DOF,ENCFF005LKD,ENCFF007LJG,ENCFF007ZBY,ENCFF008CRB,ENCFF008POE,ENCFF008UXK,ENCFF010PER,ENCFF012MMA,...,ENCFF994HKO,ENCFF994NAN,ENCFF995AUL,ENCFF995HXY,ENCFF995NEF,ENCFF995SOX,ENCFF996KKA,ENCFF999ARF,ENCSR094GVZ,ENCSR561FEE
0,ENSG00000001617.11,0.108465,3.592512,1.460604,1.294658,0.775396,0.128038,0.751979,0.468557,0.071959,...,0.390170,0.164727,0.365766,0.927295,0.241606,0.213908,0.036294,0.370106,1.828237,4.805510
1,ENSG00000002016.17,2.340519,2.193042,1.652928,1.038525,0.681675,1.437824,0.255887,0.395360,0.143489,...,0.214005,1.438948,0.098870,1.014555,0.084144,2.035110,0.035012,0.221818,1.674281,3.353565
2,ENSG00000002549.12,4.485837,21.873735,23.401274,17.130510,9.088026,39.570557,4.099586,2.839225,0.704257,...,3.291162,9.354498,2.301015,15.586799,0.190791,9.206588,0.040641,2.807642,15.714403,37.194588
3,ENSG00000002587.9,0.271143,1.185609,1.219834,1.163197,0.479181,0.319670,0.389680,0.515314,0.129505,...,0.272369,0.040779,0.207341,0.888133,0.153912,0.370529,0.027235,0.279492,1.012454,3.757029
4,ENSG00000003393.14,1.044819,3.530689,3.287493,2.806496,1.377950,4.496847,0.670018,0.756898,0.077521,...,0.412488,1.380379,0.344651,2.519098,0.100936,1.791834,0.021716,0.710399,2.794153,8.619059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3027,ENSG00000285901.1,4.279503,11.976713,16.916727,13.788750,6.731897,17.464447,2.812970,2.388816,0.291442,...,2.347390,3.704546,2.391807,11.550196,0.344354,3.957918,0.099851,2.147027,9.276557,33.280052
3028,ENSG00000285971.1,0.362848,0.026558,0.038379,0.073070,0.040245,0.118192,0.025134,0.175416,0.073634,...,0.004708,0.012302,0.008669,0.062474,0.022611,0.355801,0.004357,0.037556,0.018779,0.545745
3029,ENSG00000285972.1,0.020966,0.133358,0.014020,0.016302,0.013015,0.121071,0.046494,0.004681,0.123635,...,0.006954,0.239985,0.004200,0.008606,0.099287,0.085619,0.007358,0.010331,0.026392,0.124039
3030,ENSG00000285982.1,1.448651,0.454432,0.770982,0.741463,0.153360,1.734439,0.056215,0.090910,0.566974,...,0.080262,0.503757,0.095084,0.312576,0.287173,1.155330,0.087688,0.125223,0.665106,1.067132


## Шаг 2. Benchmark / метрики

Здесь объединяется логика из `only_benchmarking_clean.ipynb`:

- выравнивание `true` и `pred`
- `score_predictions(...)`
- средняя корреляция по генам
- средняя корреляция по cell types

Индексы выходов модели и сопоставление с `targets_human.txt` задаются через
полный `qnorm_id_to_targets_map.csv` и его пересечение с `targets_human`.
`MAPPING_PATH` в этой версии не используется: предикты собираются по всем
`original_id`, найденным в пересечении, а перед benchmark приводятся обратно
к qnorm `id` по `qnorm_target_id_map_df`.


In [4]:
true_df

,gene_id,ENCFF035CWS,ENCFF329ENM,ENCFF761SPP,ENCFF731CJY,ENCFF242NRP,ENCFF465EDA,ENCFF152CYS,ENCFF900QOK,ENCFF708BZY,...,ENCFF513XTK,ENCFF823TJX,ENCFF948VKA,ENCFF860LWY,ENCFF273RBB,ENCFF835PJC,ENCFF932OLF,ENCFF954DUN,ENCFF993FIL,ENCFF652AXF
0,ENSG00000000003.14,2.934869,46.202345,50.456795,35.074158,10.389969,34.148039,32.014933,26.995331,11.281719,...,0.634343,2.880500,15.460368,36.559947,5.085873,12.422901,0.946538,2.539223,0.223377,21.761953
1,ENSG00000000005.5,0.134624,0.216827,0.025672,0.015955,0.044488,0.033915,0.028163,0.034624,0.011776,...,0.070640,0.284737,0.014985,0.272482,0.013227,65.605988,0.016993,0.569425,9.894611,0.410150
2,ENSG00000000419.12,37.272576,14.699251,72.516842,19.338924,46.974020,30.507208,27.443869,29.205284,25.408026,...,17.360936,15.641546,14.978164,40.443895,29.606971,22.993209,30.090292,22.562257,6.853013,44.472219
3,ENSG00000000457.13,5.203078,7.978288,2.683380,3.852033,3.630531,5.124099,5.444062,4.037035,3.817333,...,7.229694,1.829993,2.588272,4.287024,2.749536,4.025520,5.134374,2.633152,38.057735,5.666748
4,ENSG00000000460.16,2.252812,1.504789,9.267197,3.633451,2.022705,2.765721,2.777990,6.655392,1.777660,...,4.852895,1.096285,4.767579,2.506871,1.903011,2.832889,8.683333,0.592331,106.912199,1.935054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21973,ENSG00000285976.1,21.511480,39.795995,52.357164,10.371127,44.550825,7.697286,9.051122,9.938084,52.861094,...,6.979819,2.941693,11.672213,15.758819,7.587835,5.887427,7.604613,12.513556,22.158715,66.519746
21974,ENSG00000285978.1,0.015078,0.008067,0.025672,0.015955,0.044488,0.033915,0.028163,0.034624,0.011776,...,0.070640,0.003423,0.014985,0.018954,0.013227,0.000813,0.016993,0.090877,0.223377,0.022599
21975,ENSG00000285982.1,0.015078,0.008067,0.025672,0.015955,0.044488,0.503671,0.028163,0.034624,0.011776,...,0.070640,0.003423,0.014985,0.002149,0.013227,0.000813,0.016993,0.006474,0.223377,0.022599
21976,ENSG00000285985.1,0.015078,0.008067,0.025672,0.015955,0.044488,0.033915,0.028163,0.034624,0.011776,...,0.070640,0.003423,0.014985,0.002149,0.013227,0.000813,0.016993,0.006474,0.223377,0.022599


In [5]:
pred_df

,gene_id,ENCFF004DOF,ENCFF005LKD,ENCFF007LJG,ENCFF007ZBY,ENCFF008CRB,ENCFF008POE,ENCFF008UXK,ENCFF010PER,ENCFF012MMA,...,ENCFF994HKO,ENCFF994NAN,ENCFF995AUL,ENCFF995HXY,ENCFF995NEF,ENCFF995SOX,ENCFF996KKA,ENCFF999ARF,ENCSR094GVZ,ENCSR561FEE
0,ENSG00000001617.11,0.108465,3.592512,1.460604,1.294658,0.775396,0.128038,0.751978,0.468557,0.071959,...,0.390170,0.164727,0.365766,0.927295,0.241606,0.213908,0.036294,0.370106,1.828237,4.805510
1,ENSG00000002016.17,2.340519,2.193042,1.652929,1.038525,0.681675,1.437824,0.255887,0.395360,0.143489,...,0.214005,1.438948,0.098870,1.014555,0.084144,2.035109,0.035012,0.221818,1.674281,3.353565
2,ENSG00000002549.12,4.485837,21.873735,23.401274,17.130510,9.088026,39.570557,4.099586,2.839225,0.704257,...,3.291162,9.354498,2.301015,15.586799,0.190791,9.206588,0.040641,2.807642,15.714403,37.194588
3,ENSG00000002587.9,0.271143,1.185609,1.219834,1.163197,0.479181,0.319670,0.389680,0.515314,0.129505,...,0.272369,0.040779,0.207341,0.888133,0.153912,0.370529,0.027235,0.279492,1.012454,3.757029
4,ENSG00000003393.14,1.044819,3.530689,3.287492,2.806496,1.377950,4.496847,0.670018,0.756898,0.077521,...,0.412488,1.380379,0.344651,2.519098,0.100936,1.791834,0.021716,0.710399,2.794153,8.619059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3027,ENSG00000285901.1,4.279503,11.976713,16.916727,13.788750,6.731897,17.464447,2.812970,2.388816,0.291442,...,2.347390,3.704546,2.391807,11.550196,0.344354,3.957918,0.099851,2.147026,9.276557,33.280052
3028,ENSG00000285971.1,0.362848,0.026558,0.038379,0.073070,0.040245,0.118192,0.025134,0.175416,0.073634,...,0.004708,0.012302,0.008669,0.062474,0.022611,0.355801,0.004357,0.037556,0.018779,0.545745
3029,ENSG00000285972.1,0.020966,0.133358,0.014020,0.016302,0.013015,0.121071,0.046494,0.004681,0.123635,...,0.006954,0.239985,0.004200,0.008606,0.099287,0.085619,0.007358,0.010331,0.026392,0.124039
3030,ENSG00000285982.1,1.448651,0.454432,0.770982,0.741463,0.153360,1.734439,0.056215,0.090910,0.566974,...,0.080262,0.503757,0.095084,0.312576,0.287173,1.155330,0.087688,0.125223,0.665106,1.067132


In [ ]:
true_all_qnorm_path = os.path.join(
    BORZOI_DIR,
    f"{split}_true_human_all_qnorm_id.csv",
)
pred_all_qnorm_path = os.path.join(
    BORZOI_DIR,
    "borzoi_predictions_valid_replicate0_pytorch.csv",
)

true_df = pd.read_csv(true_all_qnorm_path)
pred_df = pd.read_csv(pred_all_qnorm_path)

true_df["gene_id"] = true_df["gene_id"].astype(str)
pred_df["gene_id"] = pred_df["gene_id"].astype(str)

if "qnorm_target_id_map_df" not in dir():
    qnorm_target_id_map_df = pd.read_csv(os.path.join(BORZOI_DIR, "qnorm_id_to_targets_map.csv"))

orig_to_id_df = (
    qnorm_target_id_map_df[["original_id", "id"]]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .copy()
)

dup_original_ids = orig_to_id_df["original_id"].duplicated().sum()
if dup_original_ids:
    raise ValueError(f"Duplicate original_id -> id mappings detected: {dup_original_ids}")

orig_to_id = dict(zip(orig_to_id_df["original_id"], orig_to_id_df["id"]))

pred_df = pred_df.rename(columns=orig_to_id)

common_gene_ids = sorted(set(true_df["gene_id"]).intersection(pred_df["gene_id"]))
common_cols = sorted(set(true_df.columns[1:]).intersection(pred_df.columns[1:]))

if len(common_cols) == 0:
    print("true path:", true_all_qnorm_path)
    print("pred path:", pred_all_qnorm_path)
    print("first true cols:", true_df.columns[:10].tolist())
    print("first pred cols:", pred_df.columns[:10].tolist())
    raise ValueError("No common value columns between true and pred after renaming pred columns.")

true_aligned = (
    true_df[true_df["gene_id"].isin(common_gene_ids)][["gene_id"] + common_cols]
    .sort_values("gene_id")
    .reset_index(drop=True)
)
pred_aligned = (
    pred_df[pred_df["gene_id"].isin(common_gene_ids)][["gene_id"] + common_cols]
    .sort_values("gene_id")
    .reset_index(drop=True)
)

assert true_aligned["gene_id"].tolist() == pred_aligned["gene_id"].tolist()


def mean_corr_by_rows(true_mat: np.ndarray, pred_mat: np.ndarray):
    corrs = []
    for i in range(true_mat.shape[0]):
        t = true_mat[i]
        p = pred_mat[i]
        if len(t) < 2 or len(p) < 2:
            continue
        if np.std(t) == 0:
            continue
        corrs.append(np.corrcoef(t, p)[0, 1])
    return (float(np.mean(corrs)) if corrs else float("nan"), len(corrs))


def mean_corr_by_cols(true_mat: np.ndarray, pred_mat: np.ndarray):
    corrs = []
    for j in range(true_mat.shape[1]):
        t = true_mat[:, j]
        p = pred_mat[:, j]
        if len(t) < 2 or len(p) < 2:
            continue
        if np.std(t) == 0:
            continue
        corrs.append(np.corrcoef(t, p)[0, 1])
    return (float(np.mean(corrs)) if corrs else float("nan"), len(corrs))


def compute_metrics(true_frame: pd.DataFrame, pred_frame: pd.DataFrame, use_log1p: bool):
    true_mat = true_frame.iloc[:, 1:].to_numpy(dtype=np.float64)
    pred_mat = pred_frame.iloc[:, 1:].to_numpy(dtype=np.float64)

    if use_log1p:
        true_mat = np.log1p(true_mat)
        pred_mat = np.log1p(pred_mat)

    gene_corr, n_gene = mean_corr_by_rows(true_mat, pred_mat)
    cell_corr, n_cell = mean_corr_by_cols(true_mat, pred_mat)

    return {
        "transform": "log1p" if use_log1p else "raw",
        "avg_corr_by_genes": gene_corr,
        "genes_used": n_gene,
        "avg_corr_by_cells": cell_corr,
        "cells_used": n_cell,
    }


results_df = pd.DataFrame(
    [
        compute_metrics(true_aligned, pred_aligned, use_log1p=False),
        compute_metrics(true_aligned, pred_aligned, use_log1p=True),
    ]
)

print("true path:", true_all_qnorm_path)
print("pred path:", pred_all_qnorm_path)
print("common genes:", len(common_gene_ids))
print("common columns:", len(common_cols))
results_df


true path: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/valid_true_human_all_qnorm_id.csv
pred path: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/expression_prediction/benchmarks/borzoi/borzoi_predictions_valid_replicate0_pytorch.csv
common genes: 3032
common columns: 815


,transform,avg_corr_by_genes,genes_used,avg_corr_by_cells,cells_used
0,raw,0.012608,3032,0.162923,815
1,log1p,0.004658,3032,0.567656,815


In [7]:
true_aligned

,gene_id,ENCFF003KHL,ENCFF003QOJ,ENCFF007QAS,ENCFF007UXU,ENCFF009MEF,ENCFF010UKB,ENCFF010XLY,ENCFF011DHD,ENCFF012XRF,...,ENCFF993TGD,ENCFF994ERY,ENCFF994TIH,ENCFF995LYR,ENCFF996NCW,ENCFF996QEJ,ENCFF997ZTN,ENCFF998VKC,ENCFF999JNH,ENCFF999KLY
0,ENSG00000001617.11,20.083994,4.950431,42.439520,17.659977,9.034552,12.894117,10.743845,1.195635,65.436409,...,0.070356,1.420689,36.730982,24.113895,3.691086,3.808077,3.566548,2.464485,0.698713,8.409911
1,ENSG00000002016.17,13.344812,34.289955,7.129557,3.767242,14.968893,5.149452,14.535076,7.141234,1.865007,...,1.835778,3.764175,5.914418,11.162078,7.049588,3.471546,7.243537,4.428295,6.172334,4.815886
2,ENSG00000002549.12,21.716421,6.874324,8.184985,33.732012,10.103848,46.063327,18.829389,11.817480,47.459731,...,98.419918,10.015985,41.649357,23.635164,22.030008,56.238667,87.292047,63.521845,41.694238,28.164585
3,ENSG00000002587.9,0.138983,17.073780,0.050403,0.154893,0.847789,0.697388,0.606990,0.541751,0.108917,...,0.070356,14.160982,1.753051,0.206913,1.965406,0.488946,1.871320,0.032596,0.039203,1.275951
4,ENSG00000003393.14,6.915248,0.248936,7.457833,10.290860,4.596055,6.503312,16.442468,17.540639,5.938262,...,5.416478,5.766470,14.005146,6.700174,3.118164,8.535784,12.194405,17.354997,18.116900,10.103799
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3027,ENSG00000285901.1,0.015585,0.248936,2.456320,1.834389,0.062272,0.729785,0.777817,0.048588,0.315358,...,3.386903,0.292307,8.480152,0.028715,0.762187,1.539900,0.656287,0.032596,0.419382,2.595873
3028,ENSG00000285971.1,0.002757,0.248936,0.004839,0.009788,0.007431,0.053900,0.003287,0.001928,0.007396,...,0.070356,0.004881,0.007506,0.000710,0.016841,0.021480,0.018304,0.032596,0.039203,0.005199
3029,ENSG00000285972.1,1.185263,0.248936,0.004839,0.009788,0.007431,0.053900,0.003287,0.014891,0.080564,...,0.070356,0.063160,2.854551,0.805255,0.119145,0.021480,0.018304,0.032596,0.039203,0.238799
3030,ENSG00000285982.1,0.035145,0.248936,0.004839,0.075523,0.007431,0.053900,0.003287,0.001928,0.007396,...,0.070356,0.004881,0.007506,0.000037,0.002321,0.021480,0.018304,0.032596,0.039203,0.094813


In [ ]:
pred_aligned['ENCFF035CWS']

0        1.828237
1        1.674281
2       15.714403
3        1.012454
4        2.794153
          ...    
3027     9.276557
3028     0.018779
3029     0.026392
3030     0.665106
3031     0.009145
Name: ENCFF035CWS, Length: 3032, dtype: float64

In [12]:
true_aligned['ENCFF035CWS'].corr(pred_aligned['ENCFF035CWS'])

0.22613254276732042